# Sioux Falls: CNDP from DataFrames

Construct network, OD demand, and capacity-constraint DataFrames; solve a
baseline TAP; optimize capacities under an investment budget; and compare
the final equilibrium with the baseline.

**Setup:** from the repository root, run `pip install -e '.[examples]'`,
then `jupyter lab examples/`. Select a kernel with the package installed and
run all cells in order. This notebook includes its own Sioux Falls data and
is independent of the TAP notebook.

## 1. Create the input DataFrames

The complete benchmark values are embedded below: **24 nodes, 24 zones,
76 directed links, and total demand 360,600**. No CSV files or network
downloads are needed. These values are the project's Sioux Falls input,
sourced from [Transportation Networks for Research](https://github.com/bstabler/TransportationNetworks/tree/master/SiouxFalls).
We retain the benchmark's numeric units throughout; time values are not
relabelled as minutes.

Each link uses the BPR function
$t(f) = t_0[1 + b(f/c)^p]$. Nodes and demand labels are **one-based**.
`link_index` is **zero-based**, identifies a directed link, and remains
attached to that link if a DataFrame is sorted.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import traffic_assignment as ta

pd.set_option("display.max_columns", 16)
pd.set_option("display.precision", 4)

In [ ]:
# One row per directed link; all values are benchmark inputs.
links_df = pd.DataFrame(
    [[1, 2, 25900.20064, 6, 6, 0.15, 4, 0, 0, 1],
     [1, 3, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [2, 1, 25900.20064, 6, 6, 0.15, 4, 0, 0, 1],
     [2, 6, 4958.180928, 5, 5, 0.15, 4, 0, 0, 1],
     [3, 1, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [3, 4, 17110.52372, 4, 4, 0.15, 4, 0, 0, 1],
     [3, 12, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [4, 3, 17110.52372, 4, 4, 0.15, 4, 0, 0, 1],
     [4, 5, 17782.7941, 2, 2, 0.15, 4, 0, 0, 1],
     [4, 11, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [5, 4, 17782.7941, 2, 2, 0.15, 4, 0, 0, 1],
     [5, 6, 4947.995469, 4, 4, 0.15, 4, 0, 0, 1],
     [5, 9, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [6, 2, 4958.180928, 5, 5, 0.15, 4, 0, 0, 1],
     [6, 5, 4947.995469, 4, 4, 0.15, 4, 0, 0, 1],
     [6, 8, 4898.587646, 2, 2, 0.15, 4, 0, 0, 1],
     [7, 8, 7841.81131, 3, 3, 0.15, 4, 0, 0, 1],
     [7, 18, 23403.47319, 2, 2, 0.15, 4, 0, 0, 1],
     [8, 6, 4898.587646, 2, 2, 0.15, 4, 0, 0, 1],
     [8, 7, 7841.81131, 3, 3, 0.15, 4, 0, 0, 1],
     [8, 9, 5050.193156, 10, 10, 0.15, 4, 0, 0, 1],
     [8, 16, 5045.822583, 5, 5, 0.15, 4, 0, 0, 1],
     [9, 5, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [9, 8, 5050.193156, 10, 10, 0.15, 4, 0, 0, 1],
     [9, 10, 13915.78842, 3, 3, 0.15, 4, 0, 0, 1],
     [10, 9, 13915.78842, 3, 3, 0.15, 4, 0, 0, 1],
     [10, 11, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [10, 15, 13512.00155, 6, 6, 0.15, 4, 0, 0, 1],
     [10, 16, 4854.917717, 4, 4, 0.15, 4, 0, 0, 1],
     [10, 17, 4993.510694, 8, 8, 0.15, 4, 0, 0, 1],
     [11, 4, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [11, 10, 10000.0, 5, 5, 0.15, 4, 0, 0, 1],
     [11, 12, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [11, 14, 4876.508287, 4, 4, 0.15, 4, 0, 0, 1],
     [12, 3, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [12, 11, 4908.82673, 6, 6, 0.15, 4, 0, 0, 1],
     [12, 13, 25900.20064, 3, 3, 0.15, 4, 0, 0, 1],
     [13, 12, 25900.20064, 3, 3, 0.15, 4, 0, 0, 1],
     [13, 24, 5091.256152, 4, 4, 0.15, 4, 0, 0, 1],
     [14, 11, 4876.508287, 4, 4, 0.15, 4, 0, 0, 1],
     [14, 15, 5127.526119, 5, 5, 0.15, 4, 0, 0, 1],
     [14, 23, 4924.790605, 4, 4, 0.15, 4, 0, 0, 1],
     [15, 10, 13512.00155, 6, 6, 0.15, 4, 0, 0, 1],
     [15, 14, 5127.526119, 5, 5, 0.15, 4, 0, 0, 1],
     [15, 19, 14564.75315, 3, 3, 0.15, 4, 0, 0, 1],
     [15, 22, 9599.180565, 3, 3, 0.15, 4, 0, 0, 1],
     [16, 8, 5045.822583, 5, 5, 0.15, 4, 0, 0, 1],
     [16, 10, 4854.917717, 4, 4, 0.15, 4, 0, 0, 1],
     [16, 17, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [16, 18, 19679.89671, 3, 3, 0.15, 4, 0, 0, 1],
     [17, 10, 4993.510694, 8, 8, 0.15, 4, 0, 0, 1],
     [17, 16, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [17, 19, 4823.950831, 2, 2, 0.15, 4, 0, 0, 1],
     [18, 7, 23403.47319, 2, 2, 0.15, 4, 0, 0, 1],
     [18, 16, 19679.89671, 3, 3, 0.15, 4, 0, 0, 1],
     [18, 20, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [19, 15, 14564.75315, 3, 3, 0.15, 4, 0, 0, 1],
     [19, 17, 4823.950831, 2, 2, 0.15, 4, 0, 0, 1],
     [19, 20, 5002.607563, 4, 4, 0.15, 4, 0, 0, 1],
     [20, 18, 23403.47319, 4, 4, 0.15, 4, 0, 0, 1],
     [20, 19, 5002.607563, 4, 4, 0.15, 4, 0, 0, 1],
     [20, 21, 5059.91234, 6, 6, 0.15, 4, 0, 0, 1],
     [20, 22, 5075.697193, 5, 5, 0.15, 4, 0, 0, 1],
     [21, 20, 5059.91234, 6, 6, 0.15, 4, 0, 0, 1],
     [21, 22, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [21, 24, 4885.357564, 3, 3, 0.15, 4, 0, 0, 1],
     [22, 15, 9599.180565, 3, 3, 0.15, 4, 0, 0, 1],
     [22, 20, 5075.697193, 5, 5, 0.15, 4, 0, 0, 1],
     [22, 21, 5229.910063, 2, 2, 0.15, 4, 0, 0, 1],
     [22, 23, 5000.0, 4, 4, 0.15, 4, 0, 0, 1],
     [23, 14, 4924.790605, 4, 4, 0.15, 4, 0, 0, 1],
     [23, 22, 5000.0, 4, 4, 0.15, 4, 0, 0, 1],
     [23, 24, 5078.508436, 2, 2, 0.15, 4, 0, 0, 1],
     [24, 13, 5091.256152, 4, 4, 0.15, 4, 0, 0, 1],
     [24, 21, 4885.357564, 3, 3, 0.15, 4, 0, 0, 1],
     [24, 23, 5078.508436, 2, 2, 0.15, 4, 0, 0, 1]],
    columns=['init_node', 'term_node', 'capacity', 'length', 'free_flow_time', 'b', 'power', 'speed', 'toll', 'link_type'],
)
links_df.insert(0, "link_index", np.arange(len(links_df)))
display(links_df.head())

In [ ]:
# Rows are origins, columns are destinations.
zone_ids = range(1, 25)
demand_df = pd.DataFrame(
    [[0, 100, 100, 500, 200, 300, 500, 800, 500, 1300, 500, 200, 500, 300, 500, 500, 400, 100, 300, 300, 100, 400, 300, 100],
     [100, 0, 100, 200, 100, 400, 200, 400, 200, 600, 200, 100, 300, 100, 100, 400, 200, 0, 100, 100, 0, 100, 0, 0],
     [100, 100, 0, 200, 100, 300, 100, 200, 100, 300, 300, 200, 100, 100, 100, 200, 100, 0, 0, 0, 0, 100, 100, 0],
     [500, 200, 200, 0, 500, 400, 400, 700, 700, 1200, 1400, 600, 600, 500, 500, 800, 500, 100, 200, 300, 200, 400, 500, 200],
     [200, 100, 100, 500, 0, 200, 200, 500, 800, 1000, 500, 200, 200, 100, 200, 500, 200, 0, 100, 100, 100, 200, 100, 0],
     [300, 400, 300, 400, 200, 0, 400, 800, 400, 800, 400, 200, 200, 100, 200, 900, 500, 100, 200, 300, 100, 200, 100, 100],
     [500, 200, 100, 400, 200, 400, 0, 1000, 600, 1900, 500, 700, 400, 200, 500, 1400, 1000, 200, 400, 500, 200, 500, 200, 100],
     [800, 400, 200, 700, 500, 800, 1000, 0, 800, 1600, 800, 600, 600, 400, 600, 2200, 1400, 300, 700, 900, 400, 500, 300, 200],
     [500, 200, 100, 700, 800, 400, 600, 800, 0, 2800, 1400, 600, 600, 600, 900, 1400, 900, 200, 400, 600, 300, 700, 500, 200],
     [1300, 600, 300, 1200, 1000, 800, 1900, 1600, 2800, 0, 4000, 2000, 1900, 2100, 4000, 4400, 3900, 700, 1800, 2500, 1200, 2600, 1800, 800],
     [500, 200, 300, 1500, 500, 400, 500, 800, 1400, 3900, 0, 1400, 1000, 1600, 1400, 1400, 1000, 100, 400, 600, 400, 1100, 1300, 600],
     [200, 100, 200, 600, 200, 200, 700, 600, 600, 2000, 1400, 0, 1300, 700, 700, 700, 600, 200, 300, 400, 300, 700, 700, 500],
     [500, 300, 100, 600, 200, 200, 400, 600, 600, 1900, 1000, 1300, 0, 600, 700, 600, 500, 100, 300, 600, 600, 1300, 800, 800],
     [300, 100, 100, 500, 100, 100, 200, 400, 600, 2100, 1600, 700, 600, 0, 1300, 700, 700, 100, 300, 500, 400, 1200, 1100, 400],
     [500, 100, 100, 500, 200, 200, 500, 600, 1000, 4000, 1400, 700, 700, 1300, 0, 1200, 1500, 200, 800, 1100, 800, 2600, 1000, 400],
     [500, 400, 200, 800, 500, 900, 1400, 2200, 1400, 4400, 1400, 700, 600, 700, 1200, 0, 2800, 500, 1300, 1600, 600, 1200, 500, 300],
     [400, 200, 100, 500, 200, 500, 1000, 1400, 900, 3900, 1000, 600, 500, 700, 1500, 2800, 0, 600, 1700, 1700, 600, 1700, 600, 300],
     [100, 0, 0, 100, 0, 100, 200, 300, 200, 700, 200, 200, 100, 100, 200, 500, 600, 0, 300, 400, 100, 300, 100, 0],
     [300, 100, 0, 200, 100, 200, 400, 700, 400, 1800, 400, 300, 300, 300, 800, 1300, 1700, 300, 0, 1200, 400, 1200, 300, 100],
     [300, 100, 0, 300, 100, 300, 500, 900, 600, 2500, 600, 500, 600, 500, 1100, 1600, 1700, 400, 1200, 0, 1200, 2400, 700, 400],
     [100, 0, 0, 200, 100, 100, 200, 400, 300, 1200, 400, 300, 600, 400, 800, 600, 600, 100, 400, 1200, 0, 1800, 700, 500],
     [400, 100, 100, 400, 200, 200, 500, 500, 700, 2600, 1100, 700, 1300, 1200, 2600, 1200, 1700, 300, 1200, 2400, 1800, 0, 2100, 1100],
     [300, 0, 100, 500, 100, 100, 200, 300, 500, 1800, 1300, 700, 800, 1100, 1000, 500, 600, 100, 300, 700, 700, 2100, 0, 700],
     [100, 0, 0, 200, 0, 100, 100, 200, 200, 800, 600, 500, 700, 400, 400, 300, 300, 0, 100, 400, 500, 1100, 700, 0]],
    index=pd.Index(zone_ids, name="origin"),
    columns=pd.Index(zone_ids, name="destination"),
    dtype=float,
)
assert len(links_df) == 76 and demand_df.shape == (24, 24)
assert demand_df.to_numpy().sum() == 360600.0
display(demand_df.iloc[:6, :6])

## 2. Create an expansion-only design scenario

Here, lower bounds equal the original capacities and upper bounds allow a
20% increase. Bounds are **absolute capacities**, not added capacity. Equal
lower and upper bounds would fix a link. Every link must have one constraint.

The solver starts at lower bounds and uses
$I(c)=\theta\sum_a(c_a-l_a)$, with objective $TSTT(c)+I(c)$ and constraint
$I(c)\le B$. `theta` is a uniform investment multiplier; this example uses
`theta=5` and `budget=10000` in the solver's benchmark units. The legacy
per-link `investment_cost_param` field is not used by the current objective.

In [ ]:
expansion_fraction = 0.20
theta = 5.0
budget_limit = 10000.0

constraints_df = links_df[["link_index", "init_node", "term_node"]].copy()
constraints_df["lower_bound"] = links_df["capacity"]
constraints_df["upper_bound"] = links_df["capacity"] * (1.0 + expansion_fraction)
display(constraints_df.head())

scenario_settings = pd.Series({
    "maximum_capacity_increase_percent": expansion_fraction * 100,
    "theta": theta,
    "budget": budget_limit,
    "maximum_total_added_capacity_from_budget": budget_limit / theta,
}, name="value")
display(scenario_settings)

## 3. Solve the baseline user equilibrium

Baseline capacity equals the design lower bound, so baseline investment is
zero. We use a separate native network to keep the baseline available after
CNDP changes capacities. The source DataFrames remain unchanged.

In [ ]:
tap_options = ta.TapOptions()
tap_options.approach = "tapas"
tap_options.relative_gap_tolerance = 1e-10

baseline_network = ta.network_from_dataframes(
    "SiouxFalls-baseline", links_df, demand_df, node_index_base=1,
)
baseline = ta.solve_tap(baseline_network, approach=tap_options)
assert baseline.relative_gap < 1e-8
display(pd.Series({
    "total_travel_time": baseline.total_travel_time,
    "relative_gap": baseline.relative_gap,
    "solve_seconds": baseline.solve_seconds,
}, name="baseline"))

## 4. Align constraints and run CNDP

`link_index` keeps the capacity bounds attached to the correct directed link.
We deliberately shuffle the constraint rows below: the adapter restores native
link order and also validates the endpoint columns using one-based node IDs.

The upper level uses an NLopt COBYLA step; each objective evaluation solves
a lower-level TAP. The finite iteration budget makes this a practical example,
and does not certify a global optimum. Increase the limit to explore longer
runs. Passing the prepared constraints explicitly supplies all design input
in memory.

In [ ]:
network = ta.network_from_dataframes(
    "SiouxFalls-design", links_df, demand_df, node_index_base=1,
)
constraints = ta.constraints_from_dataframe(
    network,
    constraints_df.sample(frac=1.0, random_state=42),
    node_index_base=1,
)
pipeline = [ta.step(
    "nlopt", algorithm="LN_COBYLA", max_iterations=500, tolerance=1e-10,
)]
design = ta.solve_cndp(
    network, pipeline, constraints=constraints, approach=tap_options,
    theta=theta, budget=budget_limit,
)
final_relative_gap = float(design.network.relative_gap())
comparison = pd.DataFrame({
    "scenario": ["Baseline", "CNDP design"],
    "total_travel_time": [baseline.total_travel_time, design.total_travel_time],
    "investment": [0.0, design.budget],
    "objective": [baseline.total_travel_time, design.objective],
    "relative_gap": [baseline.relative_gap, final_relative_gap],
})
display(comparison)
print(f"CNDP elapsed time: {design.elapsed_seconds:.3f} seconds")

## 5. Inspect effective bounds, budget, and link changes

CNDP returns the final user-equilibrium flows and the effective bounds after
insensitive-link filtering. Use those bounds for feasibility checks. The
tables join by `link_index`, including baseline flows, so row ordering is explicit.
The solver changes the native network; it does not overwrite `links_df` or
`constraints_df`.

In [ ]:
final_costs = np.array([
    design.network.link(i).delay() for i in range(design.network.number_of_links)
], dtype=float)
design_values = pd.DataFrame({
    "link_index": np.arange(network.number_of_links),
    "final_capacity": design.capacities,
    "baseline_flow": baseline.flows,
    "final_flow": design.flows,
    "final_travel_time": final_costs,
    "effective_lower_bound": design.lower_bounds,
    "effective_upper_bound": design.upper_bounds,
})
link_design = links_df[["link_index", "init_node", "term_node", "capacity"]].rename(
    columns={"capacity": "initial_capacity"}
).merge(design_values, on="link_index", validate="one_to_one")
link_design["added_capacity"] = link_design.final_capacity - link_design.effective_lower_bound
link_design["investment"] = theta * link_design.added_capacity
link_design["flow_change"] = link_design.final_flow - link_design.baseline_flow
link_design["arc"] = (
    link_design.init_node.astype(str) + " → " + link_design.term_node.astype(str)
)

assert np.isfinite(design.objective)
assert final_relative_gap < 1e-8
assert np.all(design.capacities >= design.lower_bounds - 1e-7)
assert np.all(design.capacities <= design.upper_bounds + 1e-7)
assert np.all(design.flows >= -1e-9)
assert -1e-6 <= design.budget <= design.budget_upper_bound + 1e-6
np.testing.assert_allclose(link_design.investment.sum(), design.budget, rtol=1e-9, atol=1e-6)
np.testing.assert_allclose(design.objective, design.total_travel_time + design.budget, rtol=1e-10)
np.testing.assert_allclose(np.dot(design.flows, final_costs), design.total_travel_time, rtol=1e-10)
np.testing.assert_allclose(baseline_network.capacities(), links_df.capacity)
np.testing.assert_array_equal(constraints_df.lower_bound, links_df.capacity)
np.testing.assert_array_equal(constraints_df.upper_bound, links_df.capacity * (1 + expansion_fraction))
outgoing = np.bincount(link_design.init_node - 1, weights=link_design.final_flow, minlength=24)
incoming = np.bincount(link_design.term_node - 1, weights=link_design.final_flow, minlength=24)
np.testing.assert_allclose(
    outgoing - incoming,
    demand_df.sum(axis=1).to_numpy() - demand_df.sum(axis=0).to_numpy(),
    atol=1e-5,
)
display(link_design.nlargest(12, "added_capacity")[[
    "link_index", "arc", "initial_capacity", "final_capacity", "added_capacity",
    "investment", "baseline_flow", "final_flow",
]])

In [ ]:
design_summary = pd.Series({
    "budget_limit": design.budget_upper_bound,
    "budget_used": design.budget,
    "budget_used_percent": 100.0 * design.budget / design.budget_upper_bound,
    "total_added_capacity": link_design.added_capacity.sum(),
    "links_with_capacity_increase": int((link_design.added_capacity > 1e-6).sum()),
    "travel_time_reduction_percent": 100.0 * (1.0 - design.total_travel_time / baseline.total_travel_time),
    "objective_reduction_percent": 100.0 * (1.0 - design.objective / baseline.total_travel_time),
}, name="CNDP summary")
display(design_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
expanded = link_design.loc[link_design.added_capacity > 1e-6].nlargest(
    15, "added_capacity"
).sort_values("added_capacity")
if expanded.empty:
    axes[0].text(0.5, 0.5, "No capacity increases", ha="center", transform=axes[0].transAxes)
else:
    axes[0].barh(expanded.arc, expanded.added_capacity, color="#2878a5")
axes[0].set(xlabel="Added capacity (benchmark units)", ylabel="Directed link", title="Largest capacity increases")
positions = np.arange(len(comparison))
axes[1].bar(positions, comparison.total_travel_time, label="Total travel time", color="#2878a5")
investment_bars = axes[1].bar(
    positions, comparison.investment, bottom=comparison.total_travel_time,
    label="Investment", color="#d58a32",
)
axes[1].bar_label(investment_bars, labels=[f"Investment: {value:,.0f}" for value in comparison.investment], padding=4)
axes[1].set_ylim(0, comparison.objective.max() * 1.15)
axes[1].set_xticks(positions, labels=comparison.scenario)
axes[1].set(ylabel="Objective (solver units)", title="Travel time plus investment")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
plt.show()

## 6. Explore another design scenario

Change `expansion_fraction`, `theta`, `budget_limit`, or the pipeline iteration
limit, then rerun from the constraints cell. Each run constructs a fresh
design network. For selected links only, set `upper_bound = lower_bound` on
the others. To change demand, edit `demand_df` and rerun the baseline as well
so the two scenarios use identical demand.

`link_design` and `design_summary` remain available as pandas objects for
further analysis. All solver inputs in this notebook were supplied from the
DataFrames created above.